In [1]:
require("data.table")

# 1. Lista con los nombres de las carpetas de tus 5 experimentos
exp_folders <- c("WF9100729", "WF91007292", "WF91007293", "WF91007294", "WF91007295")

# Ruta base donde el notebook guarda los experimentos
base_dir <- "/content/buckets/b1/exp/"

# 2. Cargar la predicción de la primera semilla como tabla base
tb_ensemble <- fread(paste0(base_dir, exp_folders[1], "/prediccion.txt"))
setnames(tb_ensemble, "prob", paste0("prob_", exp_folders[1]))

# 3. Unir (JOIN) las probabilidades de las 4 semillas restantes por 'numero_de_cliente'
for (f in exp_folders[-1]) {
  tb_aux <- fread(paste0(base_dir, f, "/prediccion.txt"))
  tb_ensemble[tb_aux, on = "numero_de_cliente", paste0("prob_", f) := i.prob]
}

# 4. Calcular el promedio de probabilidades por cliente entre las 5 semillas
cols_probs <- paste0("prob_", exp_folders)
tb_ensemble[, prob_promedio := rowMeans(.SD), .SDcols = cols_probs]

# 5. Ordenar los clientes de mayor a menor probabilidad promedio
setorder(tb_ensemble, -prob_promedio)

# 6. Parámetros de envío a Kaggle
PARAM_kaggle_competencia <- "utn-2026-virtual-jr"
PARAM_kaggle_cortes <- seq(900, 2400, by = 100)

dir_salida <- paste0(base_dir, "ENSEMBLE_5SEMILLAS/")
dir.create(dir_salida, showWarnings = FALSE)

# 7. Generar los cortes y subirlos a Kaggle
for (envios in PARAM_kaggle_cortes) {
  
  tb_ensemble[, Predicted := 0L]
  tb_ensemble[1:envios, Predicted := 1L]
  
  archivo_kaggle <- paste0(dir_salida, "KA_ENSEMBLE_", envios, ".csv")
  
  # Guardar el CSV formateado para Kaggle
  fwrite(tb_ensemble[, list(numero_de_cliente, Predicted)],
    file = archivo_kaggle,
    sep = ","
  )
  
  # Ejecutar la subida por línea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM_kaggle_competencia)
  arch <- paste("-f", archivo_kaggle)
  mensaje <- paste0("-m 'Ensemble 5 semillas - cortes=", envios, "'")
  
  linea <- paste(comando, competencia, arch, mensaje)
  
  cat("Subiendo a Kaggle corte:", envios, "...\n")
  salida <- system(linea, intern = TRUE)
  cat(salida, "\n\n")
  
  Sys.sleep(12) # Pausa entre subidas
}

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%




Subiendo a Kaggle corte: 900 ...
35 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 

Subiendo a Kaggle corte: 1000 ...
34 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 

Subiendo a Kaggle corte: 1100 ...
33 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 

Subiendo a Kaggle corte: 1200 ...
32 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 

Subiendo a Kaggle corte: 1300 ...
31 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 

Subiendo a Kaggle corte: 1400 ...
30 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 

Subiendo a Kaggle corte: 1500 ...
29 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 

Subiendo a Kaggle corte: 1600 ...
28 submissions remaining today. Successfully submitted to UTN 2026 virtual jr 

Subiendo a Kaggle corte: 1700 ...
27 submissions remaining today. Successfully submitted 